# Expired Contracts

This notebook checks the unified Notion database for contracts that are still marked as Open but whose Closing Date has already passed. It then updates those records so their Contract Status becomes Expired.


In [1]:
import os
import requests
from datetime import datetime

NOTION_TOKEN = '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'
DATABASE_ID= '334701e728cb8096a94cebc0985684a2'

headers_notion = {
    "Authorization": f"Bearer {NOTION_TOKEN}",
    "Content-Type": "application/json",
    "Notion-Version": "2022-06-28",
}

In [2]:
def get_expired_open_entries():
    """
    Fetch all pages from the Notion database where:
    - Contract Status = Open
    - Closing Date is before today
    Uses pagination so it still works as the database grows.
    """
    url = f"https://api.notion.com/v1/databases/{DATABASE_ID}/query"
    all_entries = []
    today = datetime.today().date().isoformat()

    payload = {
        "filter": {
            "and": [
                {
                    "property": "Contract Status",
                    "select": {"equals": "Open"}
                },
                {
                    "property": "Closing Date",
                    "date": {"before": today}
                }
            ]
        }
    }

    while True:
        response = requests.post(url, headers=headers_notion, json=payload)

        if response.status_code != 200:
            print(f"Error fetching database: {response.status_code} - {response.text}")
            return []

        data = response.json()
        all_entries.extend(data.get("results", []))

        next_cursor = data.get("next_cursor")
        if not next_cursor:
            break

        payload["start_cursor"] = next_cursor

    return all_entries


def update_page(page_id, data):
    """
    Update a specific Notion page.
    """
    url = f"https://api.notion.com/v1/pages/{page_id}"
    payload = {"properties": data}

    response = requests.patch(url, headers=headers_notion, json=payload)

    if response.status_code == 200:
        print(f"Page {page_id} updated successfully.")
    else:
        print(f"Error updating page {page_id}: {response.status_code} - {response.text}")


def update_expired_contracts():
    """
    Find all open contracts with a past Closing Date and mark them as Expired.
    """
    entries = get_expired_open_entries()
    print(f"Found {len(entries)} open contracts that are now expired.")

    for entry in entries:
        page_id = entry["id"]

        update_data = {
            "Contract Status": {
                "select": {"name": "Expired"}
            }
        }

        update_page(page_id, update_data)

    print("Finished updating expired contracts.")

In [3]:
update_expired_contracts()

Error fetching database: 401 - {"object":"error","status":401,"code":"unauthorized","message":"API token is invalid.","request_id":"9b165ec9-8a38-4e30-b4ef-18558197b689"}
Found 0 open contracts that are now expired.
Finished updating expired contracts.
